# Week 4 - ML2 - Exercise 4: Query Rewriting + RAG Evaluation

Weeks 2 and 3 built RAG systems for a familiar document collection. Week 4 changes the domain and asks a more exam-like question: can you compare RAG variants systematically?

You will work with fictional urban climate adaptation briefs. The main goal is to compare baseline retrieval with query rewriting, then inspect whether the generated answer is grounded in the retrieved context.

## Learning Goals

- Reuse the RAG pipeline pattern in a new domain
- Use Gemini embeddings through the modern `google-genai` SDK
- Implement query rewriting for retrieval
- Evaluate retrieval with a small inspectable test set
- Use an LLM judge carefully as a lightweight grounding check

In [ ]:
from dotenv import load_dotenv
from pathlib import Path
import os
import re

import faiss
import numpy as np
import pandas as pd
from PyPDF2 import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from google import genai
from google.genai import types

load_dotenv()

## Step 0 - Gemini Setup

This notebook uses Gemini for embeddings and for short LLM calls. Set `GOOGLE_API_KEY` in your `.env` file before running the API cells.

In [ ]:
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise RuntimeError("Missing GOOGLE_API_KEY. Add it to your .env file before running this notebook.")

gemini_client = genai.Client(api_key=GOOGLE_API_KEY)

# Model settings are defined in the notebook. The .env file should only store secrets such as API keys.
GEMINI_EMBEDDING_MODEL = "gemini-embedding-2"
GEMINI_GENERATION_MODEL = "gemini-3.1-flash-lite-preview"

print("Embedding model:", GEMINI_EMBEDDING_MODEL)
# The embedding dimension is inferred from the returned vectors below.
print("Generation model:", GEMINI_GENERATION_MODEL)

## Step 1 - Load PDFs

Each document contains a `Source ID`. Later, the retrieval evaluation checks whether the expected source appears in the top retrieved chunks.

In [ ]:
def load_pdf_documents(data_dir: str = "data"):
    documents = []

    for pdf_path in sorted(Path(data_dir).glob("*.pdf")):
        reader = PdfReader(str(pdf_path))
        page_texts = [page.extract_text() or "" for page in reader.pages]
        text = "\n".join(page_texts)

        match = re.search(r"Source ID:\s*([a-z0-9_]+)", text)
        source_id = match.group(1) if match else pdf_path.stem

        documents.append({
            "source_id": source_id,
            "file_name": pdf_path.name,
            "text": text,
        })

    return documents


documents = load_pdf_documents("data")

print("Documents loaded:", len(documents))
for doc in documents:
    print(f"- {doc['source_id']} | {doc['file_name']} | {len(doc['text'])} characters")

## Step 2 - Chunk Documents (`TODO 1`)

Choose chunk settings that keep each chunk focused but still readable. Keep source metadata attached to every chunk.

In [ ]:
# TODO 1
# Set CHUNK_SIZE and CHUNK_OVERLAP.

CHUNK_SIZE =
CHUNK_OVERLAP =

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

chunks = []

for doc in documents:
    split_texts = splitter.split_text(doc["text"])
    for chunk_number, chunk_text in enumerate(split_texts):
        chunks.append({
            "chunk_id": f"{doc['source_id']}__chunk_{chunk_number:02d}",
            "source_id": doc["source_id"],
            "file_name": doc["file_name"],
            "chunk_text": chunk_text,
        })

print("Chunks created:", len(chunks))
print("First chunk ID:", chunks[0]["chunk_id"])
print(chunks[0]["chunk_text"][:500])

## Step 3 - Embed Chunks With Gemini

The Gemini-specific embedding helper is provided. Documents use `RETRIEVAL_DOCUMENT`; queries use `RETRIEVAL_QUERY`.

In [ ]:
def embed_texts_with_gemini(texts, task_type: str, batch_size: int = 16):
    vectors = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]

        result = gemini_client.models.embed_content(
            model=GEMINI_EMBEDDING_MODEL,
            contents=batch,
            config=types.EmbedContentConfig(
                task_type=task_type,
            ),
        )

        vectors.extend([embedding.values for embedding in result.embeddings])

    vectors = np.array(vectors, dtype="float32")
    faiss.normalize_L2(vectors)
    return vectors


sample_vector = embed_texts_with_gemini(
    ["cooling centers during heat alerts"],
    task_type="RETRIEVAL_QUERY",
)
print("Sample query vector shape:", sample_vector.shape)
print("Vector norm:", np.linalg.norm(sample_vector[0]))

## Step 4 - Build FAISS Retrieval (`TODO 2`)

Because vectors are normalized, inner product behaves like cosine similarity.

In [ ]:
chunk_texts_for_embedding = [
    f"Source: {chunk['source_id']}\n{chunk['chunk_text']}"
    for chunk in chunks
]

chunk_embeddings = embed_texts_with_gemini(
    chunk_texts_for_embedding,
    task_type="RETRIEVAL_DOCUMENT",
)

faiss_index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
faiss_index.add(chunk_embeddings)

print("Chunk embeddings shape:", chunk_embeddings.shape)
print("FAISS index size:", faiss_index.ntotal)

In [ ]:
# TODO 2
# Implement retrieval for one query.

def retrieve_chunks(query: str, top_k: int = 3):
    query_vector = embed_texts_with_gemini(
        [query],
        task_type="RETRIEVAL_QUERY",
    )
    k =
    scores, indices =

    results = []
    for score, index_position in zip(scores[0], indices[0]):
        item = chunks[int(index_position)].copy()
        item["similarity"] = float(score)
        results.append(item)

    return results


TOP_K = 3
test_query = "Which city uses cooling centers during heat alerts?"

for item in retrieve_chunks(test_query, top_k=TOP_K):
    print(f"{item['chunk_id']} | {item['similarity']:.4f}")
    print(item["chunk_text"][:250], "\n")

## Step 5 - Evaluation Set

This small evaluation set is intentionally inspectable. A hit means at least one expected source appeared in the final top-k retrieved chunks.

In [ ]:
evaluation_set = [
    {
        "question": "Which city uses libraries and sports halls as cooling centers during heat alerts?",
        "expected_sources": ["sunford_heat_action"],
    },
    {
        "question": "What does backup bus route B-12 do if tram service fails?",
        "expected_sources": ["northbridge_transport_resilience"],
    },
    {
        "question": "Which city plans permeable pavement near a museum?",
        "expected_sources": ["lakemont_green_roofs"],
    },
    {
        "question": "What flood warning technology does Rivergate add near the Alder River?",
        "expected_sources": ["rivergate_flood_resilience"],
    },
    {
        "question": "What is the difference between heat mitigation and heat response?",
        "expected_sources": ["climate_adaptation_glossary", "sunford_heat_action"],
    },
    {
        "question": "How does social vulnerability affect climate warning systems?",
        "expected_sources": ["climate_adaptation_glossary"],
    },
]

print("Evaluation questions:")
for example in evaluation_set:
    print("-", example["question"])
    print("  expected source(s):", ", ".join(example["expected_sources"]))

## Step 6 - Baseline Retrieval Evaluation (`TODO 3`)

We start by evaluating retrieval without query rewriting. Later we compare this baseline with the rewritten queries.

In [ ]:
# TODO 3
# Evaluate the plain user questions without query rewriting.

baseline_rows = []

for example in evaluation_set:
    question = example["question"]
    expected_sources = example["expected_sources"]

    retrieved_items =
    retrieved_sources = [item["source_id"] for item in retrieved_items]
    hit =

    baseline_rows.append({
        "strategy": "baseline",
        "question": question,
        "expected_sources": expected_sources,
        "retrieved_sources": retrieved_sources,
        "top_chunk_ids": [item["chunk_id"] for item in retrieved_items],
        "hit": hit,
    })

baseline_results = pd.DataFrame(baseline_rows)
baseline_results[["strategy", "question", "hit", "retrieved_sources"]]

## Step 7 - Query Rewrite With Gemini (`TODO 4`)

A rewrite should make the query more retrieval-friendly while preserving the user's intent.

In [ ]:
def generate_text(prompt: str, temperature: float = 0.1):
    response = gemini_client.models.generate_content(
        model=GEMINI_GENERATION_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=temperature),
    )
    return response.text.strip()


# TODO 4
# Complete the prompt template.

REWRITE_PROMPT_TEMPLATE = """
Rewrite the user question as a focused search query for urban climate adaptation policy briefs.

Rules:
- Preserve the original meaning.
- Add useful domain terms only if they are implied by the question.
- Return only the rewritten query.

User question:
{question}

Rewritten query:
"""


def rewrite_query(question: str):
    prompt = REWRITE_PROMPT_TEMPLATE.format(question=question)
    return generate_text(prompt, temperature=0.1)


example_question = evaluation_set[0]["question"]
print("Original:", example_question)
print("Rewrite:", rewrite_query(example_question))

In [ ]:
rewrite_rows = []

for example in evaluation_set:
    question = example["question"]
    expected_sources = example["expected_sources"]
    rewritten_query = rewrite_query(question)

    retrieved_items = retrieve_chunks(rewritten_query, top_k=TOP_K)
    retrieved_sources = [item["source_id"] for item in retrieved_items]
    hit = any(source in retrieved_sources for source in expected_sources)

    rewrite_rows.append({
        "strategy": "rewrite",
        "question": question,
        "transformed_queries": [rewritten_query],
        "expected_sources": expected_sources,
        "retrieved_sources": retrieved_sources,
        "top_chunk_ids": [item["chunk_id"] for item in retrieved_items],
        "hit": hit,
    })

rewrite_results = pd.DataFrame(rewrite_rows)
rewrite_results[["strategy", "question", "hit", "retrieved_sources"]]

## Step 8 - Compare Retrieval Results

Compare the baseline and rewritten-query results. A higher hit rate means the expected source appeared in the top retrieved chunks more often.

In [ ]:
print("Hit rates:")
for strategy_name, results in [
    ("baseline", baseline_results),
    ("rewrite", rewrite_results),
]:
    hit_rate = results["hit"].mean()
    print(f"{strategy_name}: {hit_rate:.2f}")

all_retrieval_results = pd.concat(
    [baseline_results, rewrite_results],
    ignore_index=True,
)

all_retrieval_results[["strategy", "question", "hit", "retrieved_sources"]]

In [ ]:
missed_results = all_retrieval_results[all_retrieval_results["hit"] == False]
missed_results[["strategy", "question", "expected_sources", "retrieved_sources"]]

## Step 9 - Generate a Grounded Answer (`TODO 5`)

Now use the retrieved context to answer one question. Keep the answer grounded and cite source IDs.

In [ ]:
# TODO 5
# Complete the answer prompt.

ANSWER_PROMPT_TEMPLATE = """
You are a careful climate adaptation analyst.
Answer the question using only the retrieved context.
If the context is insufficient, say that the answer is not available in the provided context.
Mention the source ID or source IDs you used.

Retrieved context:
{context}

Question:
{question}

Answer:
"""
answer_question = "How does Sunford protect people during active heat events?"
answer_search_query = rewrite_query(answer_question)
answer_sources = retrieve_chunks(answer_search_query, top_k=TOP_K)

context_parts = []
for item in answer_sources:
    context_parts.append(
        f"Source ID: {item['source_id']}\nChunk ID: {item['chunk_id']}\n{item['chunk_text']}"
    )

answer_context = "\n\n---\n\n".join(context_parts)
answer_prompt = ANSWER_PROMPT_TEMPLATE.format(context=answer_context, question=answer_question)
answer = generate_text(answer_prompt, temperature=0.1)

print("Question:", answer_question)
print("\nAnswer:\n", answer)
print("\nRetrieved sources:", [item["source_id"] for item in answer_sources])

## Step 10 - LLM-as-Judge Grounding Check (`TODO 6`)

This is not a perfect evaluator. Use it as one signal, and always inspect the retrieved context yourself.

In [ ]:
# TODO 6
# Complete the judge prompt.

JUDGE_PROMPT_TEMPLATE = """
You are checking whether an answer is grounded in retrieved context.

Labels:
- GROUNDED: all important claims are supported by the context.
- PARTLY_GROUNDED: some claims are supported, but at least one important claim is missing or vague.
- NOT_GROUNDED: the answer is mostly unsupported by the context.

Retrieved context:
{context}

Question:
{question}

Answer:
{answer}

Return one label and one short reason.
"""


judge_prompt = JUDGE_PROMPT_TEMPLATE.format(
    context=answer_context,
    question=answer_question,
    answer=answer,
)

judgement = generate_text(judge_prompt, temperature=0.0)
print(judgement)

## Submission Checklist

- [ ] Chunking settings selected
- [ ] FAISS retrieval implemented
- [ ] Baseline retrieval evaluated
- [ ] Query rewrite evaluated
- [ ] Retrieval results compared in a table
- [ ] One grounded answer generated
- [ ] LLM-as-judge grounding check completed
- [ ] Final reflection completed

## Final Reflection (max 8 lines)

1. Which retrieval strategy had the best hit rate?
2. Where did query rewriting help, if anywhere?
3. Where did query rewriting hurt, if anywhere?
4. How trustworthy was the LLM-as-judge result?
5. What would you improve before using this RAG system in a real setting?